# 06 -- Parameter stability

Paired script: `analysis/exit_simulation.py` (the giveback-guard predicates). This notebook
sweeps a REAL strategy parameter -- the V6.37-style giveback guard's `giveback_percent` --
across a small grid and reports how the guard's behaviour (trigger rate, mean R saved/lost)
changes, which is what "parameter stability" should mean: sensitivity of an actual tunable
knob, not just which rolling calendar window happens to contain which trades.

**Fixed, 2026-07-21 Codex review finding:** this notebook previously varied
`walk_forward.py`'s rolling calendar windows (train/test date ranges), which is not a
strategy PARAMETER at all, and produced no parameter grid or stability map.

**Uses clearly-labelled SYNTHETIC R-paths.** Real-data run: PENDING.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from analysis.exit_simulation import simulate_giveback_path

In [ ]:
# A handful of synthetic R-paths (chronological R-multiple sequences per
# trade) with varied peak/pullback shapes, so the parameter sweep below
# has something genuinely different to react to at each giveback_percent.
synthetic_r_paths = [
    [0.5, 1.0, 2.0, 1.5, 0.8],   # a moderate pullback after a 2.0R peak
    [0.3, 1.5, 3.0, 2.5, 2.0],   # a shallow pullback after a 3.0R peak
    [0.5, 1.0, 1.8, 0.3, -0.2],  # a deep pullback after a modest peak
]

GIVEBACK_PERCENT_GRID = [40.0, 60.0, 80.0]

rows = []
for giveback_percent in GIVEBACK_PERCENT_GRID:
    n_triggered = 0
    r_diffs = []
    for path in synthetic_r_paths:
        result = simulate_giveback_path(path, "v637", arm_rr=1.25, giveback_percent=giveback_percent,
                                          close_trigger_floor_r=0.05)
        if result is not None:
            _, trigger_r = result
            n_triggered += 1
            r_diffs.append(trigger_r - path[-1])  # positive = guard would have helped
    rows.append({
        "giveback_percent": giveback_percent,
        "n_triggered": n_triggered,
        "n_total": len(synthetic_r_paths),
        "mean_r_diff_when_triggered": (sum(r_diffs) / len(r_diffs)) if r_diffs else None,
    })

stability_table = pd.DataFrame(rows)
print(stability_table)

# A tighter giveback_percent (locks in profit sooner) must never trigger
# on STRICTLY FEWER paths than a looser one, for this monotonic set of
# paths -- a basic sanity/stability property of the parameter sweep
# itself, not just a print statement.
assert stability_table.iloc[0]["n_triggered"] >= stability_table.iloc[-1]["n_triggered"]

## Real-data run: PENDING

A meaningful parameter-stability read needs many real R-paths (real trade histories with
real bar-level MFE tracking), which do not exist yet -- see `analyse_giveback.py`'s own
"Real-data run: PENDING" note (notebook 02).